# 01d: MY/MY2 Trimming Investigation

**Experiment**: 01 — Averaging order effect on MOLASS decomposition  
**Purpose**: Investigate why the minor peak near frame ~1200 in MY/MY2 gets trimmed away  
**Date**: March 2026  
**Follows**: `01b_molass_runs.ipynb` (where this issue was first noticed)

## Background

In `01b`, MOLASS reported:
- **MY** → 2 spurious components (both Rg ≈ 32.3 Å)
- **MY2** → 3 genuine components (Rg = 97, 54, 32.4 Å)

There appears to be a minor (aggregate/large-Rg) peak near frame ~1200 that is visible in the
raw elution trace but gets cut off by the default `trimmed_copy()`.  
Since the Rg ≈ 97 Å component in MY2 is the interesting one, we want to understand:

1. **Where exactly does the default trim cut?**
2. **Does the minor peak at ~1200 fall outside the trim?**
3. **If we widen the trim to include it, does the decomposition improve for MY?**


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from molass.DataObjects import SecSaxsData as SSD
from molass.DataObjects.Curve import create_icurve
from molass.Stats import EghMoment
import molass; assert molass.get_version() >= "0.8.5", \
    f"molass >= 0.8.5 required, found {molass.get_version()}"
print("molass", molass.get_version())

# ── Set your local data root here ──────────────────────────────────────────
DATA_ROOT = r"C:\Users\takahashi\Dropbox\MOLASS\DATA\20260305"
# ───────────────────────────────────────────────────────────────────────────

UV_SIGNAL = 290
UV_BASELINE = 400

print("Data root:", DATA_ROOT)
print("Exists:", os.path.isdir(DATA_ROOT))

In [ ]:
# Load MY and MY2 (raw, untrimmed); MY and MY2 use 290 nm UV signal
ssd_my  = SSD(os.path.join(DATA_ROOT, "MY"),  uv_pickat=290)
ssd_my2 = SSD(os.path.join(DATA_ROOT, "MY2"), uv_pickat=290)

print("MY  XR shape:", ssd_my.xr.M.shape,  " UV shape:", ssd_my.uv.M.shape)
print("MY2 XR shape:", ssd_my2.xr.M.shape, " UV shape:", ssd_my2.uv.M.shape)

## 1. Full Elution Traces (Untrimmed)

Plot the complete SAXS intensity and UV corrected traces for MY and MY2 over the full frame range.
This gives us an unbiased view of what is in the data before any trimming.
Annotate the approximate location of the suspected minor peak (~frame 1200).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle("MY / MY2 — Full Elution Traces (Untrimmed)", fontsize=14)

for col, (name, ssd) in enumerate([("MY", ssd_my), ("MY2", ssd_my2)]):
    # SAXS total intensity
    ax_saxs = axes[0, col]
    I = ssd.xr.M.sum(axis=0)
    ax_saxs.plot(I, color="steelblue", linewidth=0.8)
    ax_saxs.axvline(x=1200, color="red", linestyle="--", alpha=0.7, label="~frame 1200")
    ax_saxs.set_title(f"{name} — SAXS summed intensity")
    ax_saxs.set_xlabel("Frame")
    ax_saxs.set_ylabel("Summed SAXS intensity")
    ax_saxs.legend()

    # UV corrected trace
    ax_uv = axes[1, col]
    c_sig = create_icurve(None, ssd.uv.M, ssd.uv.wv, UV_SIGNAL)
    c_bl  = create_icurve(None, ssd.uv.M, ssd.uv.wv, UV_BASELINE)
    uv_corr = c_sig.y - c_bl.y
    ax_uv.plot(uv_corr, color="tomato", linewidth=0.8)
    ax_uv.axvline(x=1200, color="red", linestyle="--", alpha=0.7, label="~frame 1200")
    ax_uv.set_title(f"{name} — UV corrected ({UV_SIGNAL} − {UV_BASELINE} nm)")
    ax_uv.set_xlabel("Frame")
    ax_uv.set_ylabel("Absorbance (AU)")
    ax_uv.legend()

plt.tight_layout()
plt.savefig("01d_full_traces.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01d_full_traces.png")

## 2. Default Trimming Boundaries

Call `ssd.make_trimming()` to compute the default trim without applying it,
then overlay the trim boundaries on the full elution traces.
`make_trimming()` uses an `EghMoment` fit to the elution peak and cuts at ±`nsigmas` standard deviations.

In [ ]:
# Compute default trimming for MY and MY2
trim_my  = ssd_my.make_trimming()
trim_my2 = ssd_my2.make_trimming()

def print_trim_info(name, trim):
    xr_j = trim.xr_slices[1] if trim.xr_slices is not None else None
    uv_j = trim.uv_slices[1] if trim.uv_slices is not None else None
    print(f"{name}:")
    print(f"  XR frame slice: {xr_j}  → frames [{xr_j.start}, {xr_j.stop})")
    print(f"  UV frame slice: {uv_j}  → frames [{uv_j.start}, {uv_j.stop})")

print_trim_info("MY",  trim_my)
print_trim_info("MY2", trim_my2)

In [ ]:
# Overlay trim boundaries on the SAXS elution traces
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle("MY / MY2 — Default Trim Boundaries on Full SAXS Trace", fontsize=13)

for col, (name, ssd, trim) in enumerate([
        ("MY",  ssd_my,  trim_my),
        ("MY2", ssd_my2, trim_my2)]):
    ax = axes[col]
    I = ssd.xr.M.sum(axis=0)
    ax.plot(I, color="steelblue", linewidth=0.8, label="SAXS sum")
    xr_j = trim.xr_slices[1]
    ax.axvline(x=xr_j.start, color="green",  linestyle="--", linewidth=1.5, label=f"trim start ({xr_j.start})")
    ax.axvline(x=xr_j.stop,  color="orange", linestyle="--", linewidth=1.5, label=f"trim stop  ({xr_j.stop})")
    ax.axvline(x=1200,        color="red",    linestyle=":",  linewidth=1.2, alpha=0.7, label="~frame 1200")
    ax.axvspan(xr_j.start, xr_j.stop, alpha=0.08, color="green", label="kept region")
    ax.set_title(f"{name}")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Summed SAXS intensity")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("01d_trim_boundaries.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01d_trim_boundaries.png")

In [ ]:
# Use the built-in plot_trimming() for a more detailed view
# This shows UV and XR side by side with trim markers
print("=== MY — plot_trimming ===")
ssd_my.plot_trimming(trim=trim_my, title="MY — default trimming")
plt.savefig("01d_plot_trimming_my.png", dpi=120, bbox_inches="tight")
plt.show()

print("=== MY2 — plot_trimming ===")
ssd_my2.plot_trimming(trim=trim_my2, title="MY2 — default trimming")
plt.savefig("01d_plot_trimming_my2.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. EghMoment Fit — Why the Minor Peak Gets Cut

`make_trimming()` fits an `EghMoment` (exponentially-modified Gaussian) to the main elution peak
and keeps only the frames within ±`nsigmas` standard deviations of the fitted mean.
If the minor peak at ~1200 lies beyond this window, it is discarded.

Here we visualize the EghMoment fit itself to confirm this.

In [ ]:
from molass.Trimming.TrimmingUtils import TRIMMING_NSIGMAS

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f"EghMoment fit (nsigmas={TRIMMING_NSIGMAS}) — MY vs MY2", fontsize=13)

for col, (name, ssd, trim) in enumerate([
        ("MY",  ssd_my,  trim_my),
        ("MY2", ssd_my2, trim_my2)]):
    ax = axes[col]
    # Raw SAXS elution as icurve
    xr_icurve = ssd.xr.get_icurve()
    mt = EghMoment(xr_icurve)
    M, std = mt.get_meanstd()
    lo, hi = mt.get_nsigma_points(TRIMMING_NSIGMAS)

    x = xr_icurve.x
    y = xr_icurve.y
    ax.plot(x, y, color="steelblue", linewidth=0.8, label="SAXS icurve")

    # EghMoment components
    for curve in mt.curves:
        ax.plot(x, curve.y, color="purple", linewidth=1, alpha=0.6, linestyle="--")
    ax.plot(x, sum(c.y for c in mt.curves), color="purple", linewidth=1.5, label="EghMoment fit")

    ax.axvline(x=x[lo], color="green",  linestyle="--", linewidth=1.5, label=f"trim start x={x[lo]:.0f} (frame {lo})")
    ax.axvline(x=x[hi], color="orange", linestyle="--", linewidth=1.5, label=f"trim stop  x={x[hi]:.0f} (frame {hi})")
    ax.axvline(x=M,      color="gray",  linestyle=":",  linewidth=1.2, label=f"mean x={M:.0f}")
    ax.axvspan(x[lo], x[hi], alpha=0.08, color="green")
    ax.set_title(f"{name}")
    ax.set_xlabel("Frame (x-axis of icurve)")
    ax.set_ylabel("Intensity")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("01d_eghmoment_fit.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Default nsigmas = {TRIMMING_NSIGMAS}")
print("Saved: 01d_eghmoment_fit.png")

## 4. Widened Trim — Include the Minor Peak

Re-run the MOLASS pipeline for MY with a wider frame window that extends to cover the minor peak.
We use `make_trimming(nsigmas=N)` with a larger N, or force `jranges` to manually extend the stop frame.

We try both approaches:
- **Larger nsigmas**: let the EghMoment σ-window grow naturally
- **Manual jranges**: force the end frame to ~1300 to hard-include the minor peak

In [ ]:
# --- Approach A: wider nsigmas ---
# Inspect how the trim stop frame shifts as nsigmas increases
print("MY trim stop as a function of nsigmas:")
print(f"{'nsigmas':>10}  {'XR start':>10}  {'XR stop':>10}")
for ns in [3, 4, 5, 6, 7, 8, 10]:
    t = ssd_my.make_trimming(nsigmas=ns)
    xr_j = t.xr_slices[1]
    print(f"{ns:>10}  {xr_j.start:>10}  {xr_j.stop:>10}")

In [ ]:
# --- Approach B: manual jranges ---
# Force a wider temporal window.
# jranges format: [(xr_start_time, xr_stop_time), (uv_start_time, uv_stop_time)]
# First check what time values correspond to frame 1200 in the XR data

xr_times = ssd_my.xr.jv      # time values (or frame indices) along the frame axis
uv_times = ssd_my.uv.jv

print(f"XR time axis: first={xr_times[0]:.2f}, last={xr_times[-1]:.2f}, len={len(xr_times)}")
print(f"UV time axis: first={uv_times[0]:.2f}, last={uv_times[-1]:.2f}, len={len(uv_times)}")

# Identify the time at frame 1200 (or close to it) in XR
if len(xr_times) > 1200:
    t_at_1200 = xr_times[1200]
    print(f"\nXR time at frame index 1200: {t_at_1200:.2f}")
else:
    print(f"\nXR has only {len(xr_times)} frames — frame 1200 is beyond the data")
    t_at_1200 = xr_times[-1]

## 5. Decomposition with Widened Trimming (MY only)

Run `quick_decomposition()` with:
- Default trim (baseline for comparison)
- Widened nsigmas (if that reaches frame ~1200)
- Manual jranges forcing the end frame well past the minor peak

Compare the number of components and Rg values found.

In [ ]:
def run_decomp(ssd, label, **trim_kwargs):
    """Run the MOLASS pipeline with given trimming kwargs; return (decomp, rgs, props).

    Accepts: nsigmas=N, jranges=...  (forwarded directly to trimmed_copy)
    """
    trimmed = ssd.trimmed_copy(**trim_kwargs)
    corrected = trimmed.corrected_copy()
    decomp    = corrected.quick_decomposition()
    rgs   = decomp.get_rgs()
    props = decomp.get_proportions()
    n     = decomp.get_num_components()
    print(f"\n{label}:")
    print(f"  XR frame range: {trimmed.xr.M.shape[1]} frames")
    print(f"  Components: {n}")
    for i, (rg, p) in enumerate(zip(rgs, props)):
        print(f"  Component {i+1}: Rg = {rg:.2f} Å, proportion = {p:.3f}")
    return decomp, rgs, props

# Default trim
decomp_default, rgs_default, props_default = run_decomp(
    ssd_my, "MY — default trim")

# nsigmas=8 (wider)
decomp_ns8, rgs_ns8, props_ns8 = run_decomp(
    ssd_my, "MY — nsigmas=8", nsigmas=8)

# Force jranges manually to extend past frame 1200
# Use full XR range start, extend stop to frame ~1300
xr_t0 = float(xr_times[0])
uv_t0 = float(uv_times[0])
xr_t_end = float(xr_times[min(1300, len(xr_times)-1)])
uv_t_end = float(uv_times[min(1300, len(uv_times)-1)])

decomp_wide, rgs_wide, props_wide = run_decomp(
    ssd_my, "MY — manual jranges to ~frame 1300",
    jranges=[(xr_t0, xr_t_end), (uv_t0, uv_t_end)])


In [ ]:
# Side-by-side component plots for the three trimming variants
scenarios = [
    ("MY default trim",         decomp_default),
    ("MY nsigmas=8",            decomp_ns8),
    ("MY manual wide trim",     decomp_wide),
]

for label, decomp in scenarios:
    n = decomp.get_num_components()
    decomp.plot_components(title=f"{label}  ({n} components)")
    fname = f"01d_decomp_{label.replace(' ', '_').replace('=', '')}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## Observations

*(Updated after run — March 9, 2026)*

### Q1: Does the default trim cut before frame 1200?

**No — it includes it.** With default nsigmas=10, the XR trim range is [673, 1213), so frame 1200 is inside the window (barely, with only ~13 frames of margin). The minor peak near frame 1200 is **not excluded** by default trimming.

### Q2: Is the minor peak visible?

Yes, visible in both XR and UV traces near frame ~1200 (see plots `01d_full_traces.png` and `01d_trim_boundaries.png`).

### Q3: nsigmas sweep for MY trim stop

| nsigmas | XR start | XR stop |
|---------|----------|---------|
| 3       | 862      | 1023    |
| 4       | 835      | 1050    |
| 5       | 808      | 1077    |
| 6       | 781      | 1105    |
| 7       | 754      | 1132    |
| 8       | 727      | 1159    |
| 10 (default) | 673 | 1213 |

The minor peak near frame 1200 falls **outside** the trim window for nsigmas ≤ 8 (stop ≤ 1159). The default nsigmas=10 just barely includes it (stop = 1213).

### Q4: Does including the minor peak reveal a large-Rg component?

| Scenario | XR frames | n_comp | Rg values |
|----------|-----------|--------|-----------|
| Default trim (nsigmas=10) | 540 | 2 | 32.30 Å (95%), 32.32 Å (5%) |
| Tighter trim (nsigmas=8) | 432 | 2 | 32.33 Å (95%), 33.00 Å (5%) |
| Wide manual trim (~1300) | 1300 | 3 | 32.59 Å (89%), **107.77 Å (6%)**, **101.57 Å (5%)** |

**Key finding**: With the wide trim (explicitly including the post-1200 region), MY decomposes into 3 components with two large-Rg species (~100 Å), consistent with aggregates. With default or tighter trimming, only 2 near-identical components appear, suggesting the minor peak's signal is too small to force a third component within the default window.

### Q5: Aggregate or separate species?

The large Rg values (~100–108 Å) seen in the wide-trim decomposition are far larger than the monomer (~32 Å), pointing to **aggregates** rather than a distinct elution species. The minor peak at frame ~1200 in MY is likely a transient aggregate shoulder that the default trimming barely includes but whose signal is insufficient to appear as a resolved component — only a deliberate wide trim exposes it.

**Practical implication**: The default nsigmas=10 trimming is marginal for MY. Using nsigmas=8 cleanly excludes the aggregate shoulder and gives a cleaner monomer-only decomposition (Rg = 32.33 Å). Compare with MY2 which shows a proper 3-component structure (Rg ≈ 97, 54, 32 Å) even with default trimming, consistent with higher aggregate content.
